In [2]:
"""HipMRI dataset discovery and patient split checks."""

from pathlib import Path
import re


# Extract the patient ID while checking the expected filename structure.
IMAGE_PATTERN = re.compile(
    r"case_(\d+)_week_\d+_slice_\d+\.nii\.gz"
)
SPLITS = ("train", "validate", "test")


def find_pairs(data_dir, split):
    """Return matching image and mask paths for one supplied split."""
    if split not in SPLITS:
        raise ValueError(f"Unknown split: {split}")

    data_dir = Path(data_dir)
    image_dir = data_dir / f"keras_slices_{split}"
    mask_dir = data_dir / f"keras_slices_seg_{split}"

    # Catch incorrect paths before attempting to load any images.
    for folder in (image_dir, mask_dir):
        if not folder.is_dir():
            raise FileNotFoundError(f"Dataset folder not found: {folder}")

    images = sorted(image_dir.glob("*.nii.gz"))
    if not images:
        raise ValueError(f"No NIfTI images found in {image_dir}")

    pairs = []
    expected_masks = set()

    for image_path in images:
        if IMAGE_PATTERN.fullmatch(image_path.name) is None:
            raise ValueError(f"Unexpected image filename: {image_path.name}")

        # Pair by filename rather than relying on directory ordering.
        mask_name = image_path.name.replace("case_", "seg_", 1)
        mask_path = mask_dir / mask_name

        if not mask_path.is_file():
            raise FileNotFoundError(f"Missing mask: {mask_path}")

        pairs.append((image_path, mask_path))
        expected_masks.add(mask_name)

    # Also detect masks that have no corresponding image.
    actual_masks = {p.name for p in mask_dir.glob("*.nii.gz")}
    extra_masks = actual_masks - expected_masks
    if extra_masks:
        raise ValueError(
            f"{split}: {len(extra_masks)} masks have no matching image"
        )

    return pairs


def patient_ids(pairs):
    """Read patient IDs from the verified image filenames."""
    return {
        IMAGE_PATTERN.fullmatch(image.name).group(1)
        for image, _ in pairs
    }


def audit_splits(data_dir):
    """Check all supplied splits and reject overlapping patient IDs."""
    splits = {split: find_pairs(data_dir, split) for split in SPLITS}
    patients = {
        split: patient_ids(pairs) for split, pairs in splits.items()
    }

    for first, second in (
        ("train", "validate"),
        ("train", "test"),
        ("validate", "test"),
    ):
        overlap = patients[first] & patients[second]
        if overlap:
            raise ValueError(
                f"Patient overlap between {first} and {second}: "
                f"{sorted(overlap)}"
            )

    print(f"{'Split':<12} {'Pairs':>8} {'Patients':>10}")
    for split in SPLITS:
        print(
            f"{split:<12} {len(splits[split]):>8} "
            f"{len(patients[split]):>10}"
        )

    print("All image–mask pairs matched; no patient overlap.")
    return splits


# Use the dataset's absolute location while developing in a notebook.
data_dir = Path(
    "/lambda/nfs/comp3710-virginia/PatternAnalysis-2026/"
    "recognition/ResidualUNet_s4749642/data/keras_slices_data"
)

splits = audit_splits(data_dir)

Split           Pairs   Patients
train           11460         31
validate          660          4
test              540          3
All image–mask pairs matched; no patient overlap.


In [4]:
%pip install nibabel

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 13.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 43.6 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
import nibabel as nib
import numpy as np


def load_pair(image_path, mask_path):
    """Load and validate one 2D MRI–mask pair."""
    image_nii = nib.load(image_path)
    mask_nii = nib.load(mask_path)

    # Remove singleton dimensions, keeping the stored orientation.
    image = np.squeeze(image_nii.get_fdata(dtype=np.float32))
    mask = np.squeeze(mask_nii.get_fdata())

    if image.ndim != 2 or mask.ndim != 2:
        raise ValueError("Expected 2D image and mask")

    if image.shape != mask.shape:
        raise ValueError("Image and mask dimensions differ")

    # Check spatial metadata and numerical validity before preprocessing.
    if not np.allclose(image_nii.affine, mask_nii.affine):
        raise ValueError("Image and mask affines differ")

    if not np.isfinite(image).all() or not np.isfinite(mask).all():
        raise ValueError("Image or mask contains NaN or infinite values")

    if not np.allclose(mask, np.round(mask)) or mask.min() < 0:
        raise ValueError("Mask must contain non-negative integer labels")

    # Standardise MRI intensities; leave segmentation labels unchanged.
    image = (image - image.mean()) / (image.std() + 1e-8)
    mask = np.round(mask).astype(np.int64)

    return image, mask

ModuleNotFoundError: No module named 'nibabel'

In [ ]:
# Load the first verified training pair.
image_path, mask_path = splits["train"][0]
image, mask = load_pair(image_path, mask_path)

print("Example:", image_path.name)
print("Image:", image.shape, image.dtype)
print("Mask:", mask.shape, mask.dtype)
print("Normalised image mean:", round(float(image.mean()), 4))
print("Normalised image standard deviation:", round(float(image.std()), 4))
print("Mask labels:", np.unique(mask))